In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    auc,
)
from sklearn.inspection import permutation_importance


# =============================================================================
# 1. CONFIGURAÇÕES E CONSTANTES GLOBAIS
# =============================================================================
CAMINHO = '/content/SPDadosCriminais_GERAL.csv'

# Mesma semente do mlp.py para garantir divisões idênticas dos dados
SEED = 1000


# =============================================================================
# 2. CARREGAMENTO E LIMPEZA INICIAL DOS DADOS
# =============================================================================
df = pd.read_csv(CAMINHO, sep=',', encoding='utf-8-sig', low_memory=False)
print(f"-> Dimensões originais do dataset: {df.shape[0]} linhas e {df.shape[1]} colunas")

# Conversão numérica segura das coordenadas
df['LAT_NUM'] = pd.to_numeric(df['LATITUDE'], errors='coerce')
df['LON_NUM'] = pd.to_numeric(df['LONGITUDE'], errors='coerce')

# Remoção de coordenadas nulas ou (0, 0) inválidas
df_limpo = df[
    df['LAT_NUM'].notna() & df['LON_NUM'].notna() &
    (df['LAT_NUM'] != 0) & (df['LON_NUM'] != 0)
].copy()

# Tratamento temporal: hora e dia da semana
df_limpo['HORA_LIMPA'] = df_limpo['HORA_OCORRENCIA_BO'].astype(str).str.strip().str.split(':').str[0]
df_limpo['HORA_INT'] = pd.to_numeric(df_limpo['HORA_LIMPA'], errors='coerce').fillna(12)
df_limpo['DATA_DT'] = pd.to_datetime(df_limpo['DATA_OCORRENCIA_BO'], errors='coerce')
df_limpo['DIA_SEMANA'] = df_limpo['DATA_DT'].dt.dayofweek.fillna(0)

print(f"-> Dimensões após limpeza de coordenadas: {df_limpo.shape[0]} linhas")

# Amostragem para viabilidade computacional rápida (100.000 amostras)
if len(df_limpo) > 800000:
    df_sample = df_limpo.sample(n=800000, random_state=SEED).copy()
else:
    df_sample = df_limpo.copy()


# =============================================================================
# 3. DIVISÃO ESTRITA DE TREINO/TESTE (EVITA DATA LEAKAGE)
# =============================================================================
df_train, df_test = train_test_split(df_sample, test_size=0.30, random_state=SEED)

# Arredondamento das coordenadas para criar blocos/zonas de vizinhança
for dataset in [df_train, df_test]:
    dataset['LAT_ROUND'] = dataset['LAT_NUM'].round(2)
    dataset['LON_ROUND'] = dataset['LON_NUM'].round(2)


# =============================================================================
# 4. ENGENHARIA DO ALVO (BASEADA APENAS NO TREINO)
# =============================================================================
# Frequência de crimes por zona calculada somente no conjunto de treino
contagem_zona_treino = (
    df_train.groupby(['LAT_ROUND', 'LON_ROUND'])
    .size()
    .reset_index(name='FREQ_CRIMES')
)

# Limiar de perigo: mediana das frequências no treino
limiar_perigo = contagem_zona_treino['FREQ_CRIMES'].median()

# Mapeando de volta ao treino
df_train = pd.merge(df_train, contagem_zona_treino, on=['LAT_ROUND', 'LON_ROUND'], how='left')
df_train['LOCAL_PERIGOSO'] = (df_train['FREQ_CRIMES'] > limiar_perigo).astype(int)

# Mapeando ao teste de forma "cega" (somente com o que foi aprendido no treino)
df_test = pd.merge(df_test, contagem_zona_treino, on=['LAT_ROUND', 'LON_ROUND'], how='left')
df_test['FREQ_CRIMES'] = df_test['FREQ_CRIMES'].fillna(0)
df_test['LOCAL_PERIGOSO'] = (df_test['FREQ_CRIMES'] > limiar_perigo).astype(int)

print("\nDistribuição de Classes no Treino:", np.bincount(df_train['LOCAL_PERIGOSO']))
print("Distribuição de Classes no Teste: ", np.bincount(df_test['LOCAL_PERIGOSO']))


# =============================================================================
# 5. ENGENHARIA DE RECURSOS ESPACIAIS (PROJEÇÃO 3D)
# =============================================================================
for dataset in [df_train, df_test]:
    lat_rad = np.radians(dataset['LAT_NUM'])
    lon_rad = np.radians(dataset['LON_NUM'])
    dataset['COORD_X'] = np.cos(lat_rad) * np.cos(lon_rad)
    dataset['COORD_Y'] = np.cos(lat_rad) * np.sin(lon_rad)
    dataset['COORD_Z'] = np.sin(lat_rad)

# Mesmas features finais utilizadas no pipeline consolidado do mlp.py
features_finais = ['COORD_X', 'COORD_Y', 'COORD_Z', 'HORA_INT', 'DIA_SEMANA']

X_train = df_train[features_finais]
y_train = df_train['LOCAL_PERIGOSO']
X_test = df_test[features_finais]
y_test = df_test['LOCAL_PERIGOSO']

# Escalonamento adequado (ajustado somente no treino)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# =============================================================================
# 6. BUSCA DE HIPERPARÂMETROS (GridSearchCV)
# =============================================================================
# Espaço de busca para Regressão Logística:
#   - C: inverso da força de regularização (menor C = mais regularização)
#   - penalty: tipo de regularização (L1, L2 ou nenhuma)
#   - solver: algoritmo de otimização (liblinear suporta L1 e L2; lbfgs e
#     newton-cg suportam L2 e nenhuma; saga suporta todas)
param_grid = {
    'C': [0.01, 0.1, 1.0, 10.0, 100.0],
    'penalty': ['l2', 'l1', 'none'],
    'solver': ['liblinear', 'lbfgs', 'saga'],
}

# Combinações inválidas são ignoradas com error_score=np.nan
combinacoes_validas = []
for C in param_grid['C']:
    for penalty in param_grid['penalty']:
        for solver in param_grid['solver']:
            if penalty == 'l1' and solver in ('lbfgs',):
                continue  # lbfgs não suporta L1
            if penalty == 'none' and solver == 'liblinear':
                continue  # liblinear não suporta penalty=None
            combinacoes_validas.append({'C': [C], 'penalty': [penalty], 'solver': [solver]})

print(f"\n-> Total de {len(combinacoes_validas)} combinações válidas de hiperparâmetros serão testadas (3-fold CV).")

logreg_base = LogisticRegression(
    max_iter=1000,
    random_state=SEED,
    n_jobs=-1,
)

print("Iniciando a busca de hiperparâmetros com GridSearchCV (3-fold CV)...")
grid_search = GridSearchCV(
    estimator=logreg_base,
    param_grid=combinacoes_validas,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1,
    refit=True,
)
grid_search.fit(X_train_scaled, y_train)

print("\nMelhores parâmetros encontrados:", grid_search.best_params_)
print(f"Melhor acurácia de validação cruzada: {grid_search.best_score_:.4f}")


# =============================================================================
# 7. AVALIAÇÃO DO MODELO OTIMIZADO NO CONJUNTO DE TESTE
# =============================================================================
best_logreg = grid_search.best_estimator_

y_pred = best_logreg.predict(X_test_scaled)
y_prob = best_logreg.predict_proba(X_test_scaled)[:, 1]

print("\n" + "=" * 60)
print("DESEMPENHO DA REGRESSÃO LOGÍSTICA OTIMIZADA")
print("=" * 60)
print(f"Acurácia Final no Teste: {accuracy_score(y_test, y_pred):.4f}\n")
print("Relatório de Classificação:")
print(classification_report(y_test, y_pred))


# =============================================================================
# 8. VISUALIZAÇÕES: MATRIZ DE CONFUSÃO E CURVA ROC
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Matriz de Confusão
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Baixo Risco (0)', 'Alto Risco (1)'],
            yticklabels=['Baixo Risco (0)', 'Alto Risco (1)'])
axes[0].set_title('Matriz de Confusão - Regressão Logística')
axes[0].set_ylabel('Classe Real')
axes[0].set_xlabel('Classe Predita')

# Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC (AUC = {roc_auc:.4f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Aleatório')
axes[1].set_title('Curva ROC - Regressão Logística')
axes[1].set_xlabel('Taxa de Falsos Positivos')
axes[1].set_ylabel('Taxa de Verdadeiros Positivos')
axes[1].grid(True, linestyle='--', alpha=0.6)
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()


# =============================================================================
# 9. IMPORTÂNCIA DAS FEATURES (COEFICIENTES E PERMUTAÇÃO)
# =============================================================================
# A Regressão Logística expõe diretamente os coeficientes do modelo linear.
# Em features escalonadas, a magnitude absoluta do coeficiente indica o
# impacto relativo de cada variável na probabilidade predita.
coef_df = pd.DataFrame({
    'Feature': features_finais,
    'Coeficiente': best_logreg.coef_[0],
})
coef_df['Abs_Coeficiente'] = coef_df['Coeficiente'].abs()
coef_df = coef_df.sort_values(by='Abs_Coeficiente', ascending=False)

print("\nCoeficientes do modelo (features escalonadas):")
print(coef_df[['Feature', 'Coeficiente']].to_string(index=False))

# Permutation Importance (para permitir comparação direta com o mlp.py)
print("\nCalculando a importância das features por permutação no conjunto de teste...")
result = permutation_importance(
    best_logreg, X_test_scaled, y_test, n_repeats=10, random_state=SEED, n_jobs=-1
)

importances_df = pd.DataFrame({
    'Feature': features_finais,
    'Importance_Mean': result.importances_mean,
    'Importance_Std': result.importances_std,
}).sort_values(by='Importance_Mean', ascending=False)

print("\nImportância por permutação:")
print(importances_df.to_string(index=False))

# Plotando os dois tipos de importância lado a lado
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.barplot(x='Coeficiente', y='Feature', data=coef_df, hue='Feature',
            palette='viridis', legend=False, ax=axes[0])
axes[0].set_title('Coeficientes da Regressão Logística')
axes[0].set_xlabel('Valor do Coeficiente')
axes[0].set_ylabel('Variável')
axes[0].axvline(0, color='black', linestyle='--', lw=0.8)
axes[0].grid(True, linestyle='--', alpha=0.5)

sns.barplot(x='Importance_Mean', y='Feature', data=importances_df, hue='Feature',
            palette='magma', legend=False, ax=axes[1])
axes[1].set_title('Importância das Variáveis (Permutation Importance)')
axes[1].set_xlabel('Redução Média na Acurácia (Impacto)')
axes[1].set_ylabel('Variável')
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()


# =============================================================================
# 10. RESUMO FINAL PARA COMPARAÇÃO COM O MLP
# =============================================================================
print("\n" + "=" * 60)
print("RESUMO FINAL - REGRESSÃO LOGÍSTICA")
print("=" * 60)
print(f"Melhores hiperparâmetros : {grid_search.best_params_}")
print(f"Acurácia CV (treino)     : {grid_search.best_score_:.4f}")
print(f"Acurácia no teste        : {accuracy_score(y_test, y_pred):.4f}")
print(f"AUC no teste             : {roc_auc:.4f}")
print("=" * 60)

-> Dimensões originais do dataset: 813622 linhas e 16 colunas
-> Dimensões após limpeza de coordenadas: 706652 linhas

Distribuição de Classes no Treino: [ 16774 477882]
Distribuição de Classes no Teste:  [  8374 203622]

-> Total de 35 combinações válidas de hiperparâmetros serão testadas (3-fold CV).
Iniciando a busca de hiperparâmetros com GridSearchCV (3-fold CV)...
Fitting 3 folds for each of 35 candidates, totalling 105 fits
